In [55]:
import os
from dotenv import load_dotenv
from langsmith import wrappers
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chat_models import init_chat_model
from langsmith import traceable
from langsmith import Client
from typing_extensions import Annotated,TypedDict
from google import genai
from google.genai import types
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

True

In [29]:
os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")
os.environ["LANGSMITH_TRACING"]="true"

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [7]:
client = Client()

# define the dataset
dataset_name = "Chatbots Evaluation"
dataset = client.create_dataset(dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {"question": "What is LangChain?"},
            "outputs": {"answer": "A framework for building LLM applications"},
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
        },
        {
            "inputs": {"question": "What is OpenAI?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        },
        {
            "inputs": {"question": "What is Google?"},
            "outputs": {"answer": "A technology company known for search"},
        },
        {
            "inputs": {"question": "What is Mistral?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        }
    ]
)

{'example_ids': ['f6522a16-af48-4698-bffe-a4a9562c6488',
  '1ac7a5d3-135c-4ac6-a606-8973c9b09636',
  'de35cfb8-dedc-4d58-829d-02b3bbbf0f0b',
  '98a4e425-7fb9-4bdc-94f1-75513da3a54f',
  'd84e14df-be35-40be-a5c0-82d1a168dbff'],
 'count': 5,
 'as_of': '2026-08-17T13:03:33.4423972Z'}

### LLM as a judge

In [35]:
genai_client = genai.Client()

eval_instructions = "You are an expert professor specialized in grading students' answers to questions."

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    user_content = f"""You are grading the following question:    {inputs['question']}    
    Here is the real answer:    {reference_outputs['answer']}    
    You are grading the following predicted answer:    {outputs['response']}    
    Respond with CORRECT or INCORRECT:    Grade:    """

    full_prompt = f"System Instructions:\n{eval_instructions}\n\nTask:\n{user_content}"

    response = genai_client.models.generate_content(
        model = "gemini-3.5-flash-lite",
        contents = full_prompt,
        config = {"temperature": 0}
    )

    return "CORRECT" in response.text.strip().upper()


In [36]:
# Concisions- checks whether the actual output is less than 2x the length of the expected result.
def concision(outputs: dict, reference_outputs:dict) -> bool:
    return int(len(outputs["response"]) < 2 * len(reference_outputs["answer"]))

In [37]:
default_instructions = "Respond to the users question in a short, concise manner (one short sentence)."
def my_app(question: str, model:str = "gemini-3.5-flash-lite", instructions:str = default_instructions)-> str:

    genai_client = genai.Client()

    response = genai_client.models.generate_content(
        model=model,
        contents=question,
        config=types.GenerateContentConfig(
            system_instruction=instructions,
            temperature=0
       ),
    )
    return response.text

In [38]:
# calling my app for datapoints
def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"])}

In [39]:
# evaluations
experiment_results = client.evaluate(
    ls_target,
    data=dataset_name,
    evaluators=[correctness, concision],
    experiment_prefix="gemini-flash-chatbot"
)

View the evaluation results for experiment: 'gemini-flash-chatbot-c989ec50' at:
https://smith.langchain.com/o/2722c776-1db7-404e-b37b-fd36cd1fb1be/datasets/5ea671a1-7943-49e4-a7ed-9eba482e236a/compare?selectedSessions=2162b5e6-0815-4b6b-a6d5-9e029078801d




5it [00:10,  2.01s/it]


## Evaluation for RAG

In [46]:
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250, chunk_overlap=0
)

doc_splits = text_splitter.split_documents(docs_list)

vector_store = InMemoryVectorStore.from_documents(
    documents=doc_splits,
    embedding=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"),
)

retriever = vector_store.as_retriever(k=6)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3813.85it/s]


In [47]:
retriever.invoke("what is agents")

[Document(id='e6f2a882-fd4c-4ef4-8b7a-08c5b2411948', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, 

In [49]:
llm = init_chat_model("google_genai:gemini-3.5-flash-lite", temperature=0)
llm

ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.5.4', 'langchain': '1.3.15', 'langchain-google-genai': '4.3.4'}}, profile={'name': 'Gemini 3.5 Flash Lite', 'release_date': '2026-07-21', 'last_updated': '2026-07-21', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True, 'reasoning_effort_levels': ['minimal', 'low', 'medium', 'high'], 'reasoning_effort_default': 'minimal'}, google_api_key=SecretStr('**********'), model='gemini-3.5-flash-lite', temperature=0.0, client=<google.genai.client.Client object at 0x000001DD41DBAE10>, default_metadata=(), model_kwar

In [52]:
@traceable
def rag_bot(question:str) -> dict:
    docs = retriever.invoke(question)
    docs_string = "".join(doc.page_content for doc in docs)
    instructions = f"""You are a helpful assistant who is good at analyzing source information and answering questions.       
    Use the following source documents to answer the user's questions.       
    If you don't know the answer, just say that you don't know.       
    Use three sentences maximum and keep the answer concise.Documents:
    {docs_string}"""

    ai_msg=llm.invoke([
         {"role": "system", "content": instructions},
        {"role": "user", "content": question},

    ])
    return {"answer":ai_msg.content,"documents":docs}


In [53]:
rag_bot("What is agents")

d:\Projects\rag\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


{'answer': [{'type': 'text',
   'text': 'Based on the provided documents, agents are built with a large language model (LLM) serving as their core controller or "brain," acting as a powerful general problem solver. Examples of proof-of-concept demos include AutoGPT, GPT-Engineer, and BabyAGI. Additionally, agent systems can involve generative agents where relationships, observations, and planning are all taken into consideration.',
   'extras': {'signature': 'El4KXAERTTIPS8pCFll+gLBst6sbor46gbhBEQk3vKxtCk/r+nQf7rGdSg7kGHpJd5tWh0qZi99+b8zXkSJnlja34pcdMD76vcis0iRQ/vdUVcmGJ6tOrDbZsr31ZOBa'}}],
 'documents': [Document(id='e6f2a882-fd4c-4ef4-8b7a-08c5b2411948', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring exampl

In [54]:
client=Client()

# Define the examples for the dataset
examples = [
    {
        "inputs": {"question": "How does the ReAct agent use self-reflection? "},
        "outputs": {"answer": "ReAct integrates reasoning and acting, performing actions - such tools like Wikipedia search API - and then observing / reasoning about the tool outputs."},
    },
    {
        "inputs": {"question": "What are the types of biases that can arise with few-shot prompting?"},
        "outputs": {"answer": "The biases that can arise with few-shot prompting include (1) Majority label bias, (2) Recency bias, and (3) Common token bias."},
    },
    {
        "inputs": {"question": "What are five types of adversarial attacks?"},
        "outputs": {"answer": "Five types of adversarial attacks are (1) Token manipulation, (2) Gradient based attack, (3) Jailbreak prompting, (4) Human red-teaming, (5) Model red-teaming."},
    }
]


dataset_name="RAG Test Evaluation"
dataset = client.create_dataset(dataset_name=dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=examples
)

{'example_ids': ['944986e9-b3d6-40c0-b361-f6fce5d10319',
  '7441423e-1e8f-4bc4-876f-8f775418a6d1',
  'bf19d53c-d219-4e61-a252-7fec4a68869e'],
 'count': 3,
 'as_of': '2026-08-17T14:51:08.553025946Z'}

## Correctness

In [57]:
class Correctness_Grade(TypedDict):
    # Note that the order in the fields are defined is the order in which the model will generate them.
    # It is useful to put explanations before responses because it forces the model to think through
    # its final response before generating it:
    explanation: Annotated[str, ..., "Explain your reasoning for score"]
    correct: Annotated[bool, ..., "True if the answer is correct, False otherwise"]

correctness_instructions = """You are a teacher grading a quiz. 

                    You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and the STUDENT ANSWER. 

                    Here is the grade criteria to follow:
                    (1) Grade the student answers based ONLY on their factual accuracy relative to the ground truth answer. 
                    (2) Ensure that the student answer does not contain any conflicting statements.
                    (3) It is OK if the student answer contains more information than the ground truth answer, as long as it is factually accurate relative to the  ground truth answer.

                    Correctness:
                        A correctness value of True means that the student's answer meets all of the criteria.
                        A correctness value of False means that the student's answer does not meet all of the criteria.

                    Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

                    Avoid simply stating the correct answer at the outset."""

grader_llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", temperature=0).with_structured_output(Correctness_Grade, method="json_schema", strict=True)

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    """An evaluator for RAG answer accuracy"""
    answers = f"""\
        QUESTION: {inputs['question']}
        GROUND TRUTH ANSWER: {reference_outputs['answer']}
        STUDENT ANSWER: {outputs['answer']}"""

    # Run evaluator
    grade = grader_llm.invoke([
        {"role": "system", "content": correctness_instructions}, 
        {"role": "user", "content": answers}
    ])
    return grade["correct"]



## Relevance: Response vs input

In [58]:
class Relevance_Grade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "Provide the score on whether the answer addresses the question"]

# Grade prompt
relevance_instructions="""You are a teacher grading a quiz. 

You will be given a QUESTION and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION
(2) Ensure the STUDENT ANSWER helps to answer the QUESTION

Relevance:
A relevance value of True means that the student's answer meets all of the criteria.
A relevance value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM
relevance_llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", temperature=0).with_structured_output(Relevance_Grade, method="json_schema", strict=True)

# Evaluator
def relevance(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer helpfulness."""
    answer = f"QUESTION: {inputs['question']}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = relevance_llm.invoke([
        {"role": "system", "content": relevance_instructions}, 
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]

## Groundedness: Response vs retrieved docs

In [59]:
class Grounded_Grade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    grounded: Annotated[bool, ..., "Provide the score on if the answer hallucinates from the documents"]

# Grade prompt
grounded_instructions = """You are a teacher grading a quiz. 

You will be given FACTS and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is grounded in the FACTS. 
(2) Ensure the STUDENT ANSWER does not contain "hallucinated" information outside the scope of the FACTS.

Grounded:
A grounded value of True means that the student's answer meets all of the criteria.
A grounded value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM 
grounded_llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite",  temperature=0).with_structured_output(Grounded_Grade, method="json_schema", strict=True)

# Evaluator
def groundedness(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer groundedness."""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = grounded_llm.invoke([{"role": "system", "content": grounded_instructions}, {"role": "user", "content": answer}])
    return grade["grounded"]

## Retrieval Relevance: Retrieved docs vs input

In [60]:
class Retrieval_RelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "True if the retrieved documents are relevant to the question, False otherwise"]

# Grade prompt
retrieval_relevance_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION and a set of FACTS provided by the student. 

Here is the grade criteria to follow:
(1) You goal is to identify FACTS that are completely unrelated to the QUESTION
(2) If the facts contain ANY keywords or semantic meaning related to the question, consider them relevant
(3) It is OK if the facts have SOME information that is unrelated to the question as long as (2) is met

Relevance:
A relevance value of True means that the FACTS contain ANY keywords or semantic meaning related to the QUESTION and are therefore relevant.
A relevance value of False means that the FACTS are completely unrelated to the QUESTION.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM
retrieval_relevance_llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", temperature=0).with_structured_output(Retrieval_RelevanceGrade, method="json_schema", strict=True)

def retrieval_relevance(inputs: dict, outputs: dict) -> bool:
    """An evaluator for document relevance"""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nQUESTION: {inputs['question']}"

    # Run evaluator
    grade = retrieval_relevance_llm.invoke([
        {"role": "system", "content": retrieval_relevance_instructions}, 
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]

In [62]:
# run the evaluations
def target(inputs:dict) -> dict:
    return rag_bot(inputs["question"])

experiment_results = client.evaluate(
    target,
    data=dataset_name,
    evaluators=[correctness, groundedness, relevance, retrieval_relevance],
    experiment_prefix="rag_doc_relevance",
    metadata={"version": "LCEL context, gemini-3.5-flash-lite"}
)

experiment_results.to_pandas()

View the evaluation results for experiment: 'rag_doc_relevance-65fce9a8' at:
https://smith.langchain.com/o/2722c776-1db7-404e-b37b-fd36cd1fb1be/datasets/c98223bb-8776-42ab-b6ea-898ade7b2137/compare?selectedSessions=6110e1d5-3005-46b6-be5f-a3e08a1c97f5




0it [00:00, ?it/s]d:\Projects\rag\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
d:\Projects\rag\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
d:\Projects\rag\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
d:\Projects\rag\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = se

,inputs.question,outputs.answer,outputs.documents,error,reference.answer,feedback.correctness,feedback.groundedness,feedback.relevance,feedback.retrieval_relevance,execution_time,example_id,id
0,What are the types of biases that can arise wi...,"[{'type': 'text', 'text': 'According to Zhao e...",[page_content='Text: i'll bet the video game i...,None,The biases that can arise with few-shot prompt...,True,True,True,True,1.377527,7441423e-1e8f-4bc4-876f-8f775418a6d1,01a01043-8afc-7670-b284-404df037a495
1,How does the ReAct agent use self-reflection?,"[{'type': 'text', 'text': 'Based on the provid...",[page_content='Self-reflection is a vital aspe...,None,"ReAct integrates reasoning and acting, perform...",False,True,True,True,1.525047,944986e9-b3d6-40c0-b361-f6fce5d10319,01a01043-db39-7613-9c24-cd5b09d21f51
2,What are five types of adversarial attacks?,"[{'type': 'text', 'text': 'Based on the provid...",[page_content='Adversarial attacks are inputs ...,None,Five types of adversarial attacks are (1) Toke...,True,True,True,True,1.061523,bf19d53c-d219-4e61-a252-7fec4a68869e,01a01043-fd5f-7e52-9d5d-bf06be74ee81
